# MarketReactionLab | Historical data preparation and integrity audit

**Cleaned from the original Kaggle ingestion notebook** (`notebook4bc3158fad`, 46 troubleshooting/data-preparation cells).

This is a **documented reconstruction of the historical data workflow**, not proof that the cleaned notebook has been rerun end-to-end. The original notebook is preserved in its Kaggle history; we do not replace its results with invented outputs.

**Source:** FNSPID news and historical price files via the Hugging Face dataset `Zihan1004/FNSPID`. The code below preserves the source project's download and parsing conventions, and uses the existing `src.data` builder instead of copying 50k lines of production logic into a notebook.

**Research warning:** The archival news publication metadata is classified `untrusted_date_only`. A conservative one-session date lag reduces some risks but **does not establish verified real-world publication timestamps or zero leakage**.

**Notebook path:** `notebooks/research/06_historical_pipeline_clean.ipynb`.

**Sections:** configuration → optional downloads → source profiling → matched news → project processing → build → audit → provenance. Kaggle CPU is sufficient; full source download is very large. All expensive steps are opt-in.

## 1 · Configuration and environment

Most readers should **only run the audit section**, using existing `processed_v2_conservative` data. Set `RUN_SOURCE_DOWNLOADS=True` only if you intend to download the full original source, and check network / disk budgets first.

A checked-out copy of the project is needed for the original builder; use the exact commit corresponding to the historical experiment if available. The current default branch may have changed since training.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

import numpy as np
import pandas as pd

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
TEMP = Path("/tmp/marketreactionlab")
DATA = WORK / "data"
REPO = WORK / "MarketReactionLab"

RUN_SOURCE_DOWNLOADS = False    # downloads can be multiple GB
RUN_BUILD = False               # full dataset build is expensive
RUN_TESTS = False               # optional project regression tests
FULL_HISTORICAL_DIR = DATA / "processed_v2_conservative"

for path in [TEMP / "downloads", DATA, WORK / "reports"]:
    path.mkdir(parents=True, exist_ok=True)

print("Working directory:", WORK)
print("Source downloads enabled:", RUN_SOURCE_DOWNLOADS)
print("Dataset reconstruction enabled:", RUN_BUILD)
print("Free disk space (GB):", round(shutil.disk_usage(WORK).free / 1024**3, 2))

## 2 · Locate archived FNSPID source files

The original notebook downloaded `Stock_price/full_history.zip` and `Stock_news/All_external.csv` with `hf_hub_download`, then checked file schemas.

The `RUN_SOURCE_DOWNLOADS` switch prevents a multi-gigabyte download during an ordinary notebook review. Supply already-downloaded files through Kaggle input mounts if possible.

In [ ]:
PRICE_ZIP = TEMP / "downloads" / "Stock_price" / "full_history.zip"
NEWS_CSV = TEMP / "downloads" / "Stock_news" / "All_external.csv"

if RUN_SOURCE_DOWNLOADS:
    from huggingface_hub import hf_hub_download
    for filename in ("Stock_price/full_history.zip", "Stock_news/All_external.csv"):
        location = hf_hub_download(
            repo_id="Zihan1004/FNSPID",
            filename=filename,
            repo_type="dataset",
            local_dir=str(TEMP / "downloads")
        )
        print("Downloaded:", location)

if not PRICE_ZIP.is_file() and (DATA / "full_history.zip").is_file():
    PRICE_ZIP = DATA / "full_history.zip"
print("Price archive:", PRICE_ZIP, "| exists:", PRICE_ZIP.is_file())
print("News CSV:", NEWS_CSV, "| exists:", NEWS_CSV.is_file())

if PRICE_ZIP.is_file():
    with zipfile.ZipFile(PRICE_ZIP) as archive:
        files = [p for p in archive.namelist() if p.startswith("full_history/") and p.lower().endswith(".csv")]
        print("Stock price CSVs:", len(files))
        for ticker in ("AAPL", "MSFT", "SPY"):
            member = f"full_history/{ticker}.csv"
            if member in archive.namelist():
                with archive.open(member) as datafile:
                    sample = pd.read_csv(datafile, nrows=3)
                print(ticker, "source columns:", sample.columns.tolist())

## 3 · Profile FNSPID headline source in memory-safe chunks

This stage mirrors the **chunked** source profiling from the original notebook. It reads only known columns and stores counts, not third-party article text in this public notebook.

**Data quality note:** A parsed date is not a trusted source publication timestamp.

In [ ]:
if NEWS_CSV.is_file():
    from collections import Counter
    counts = Counter()
    total_news = 0
    for chunk in pd.read_csv(
        NEWS_CSV, usecols=["Date", "Article_title", "Stock_symbol"],
        chunksize=100_000, dtype="string", on_bad_lines="skip"
    ):
        total_news += len(chunk)
        dates = pd.to_datetime(chunk["Date"], errors="coerce", utc=True, format="mixed")
        counts.update(dates.dropna().dt.year.value_counts().to_dict())
    print("News article rows read:", f"{total_news:,}")
    print("Articles by parsed calendar year:", dict(sorted(counts.items())))
else:
    print("Skipping source profiling: large original CSV not mounted.")

## 4 · Match source symbols to price archive (opt-in)

The historical experiment streamed headline data and stored only ticker matches as Parquet. This is a **source-level preprocess**, not an independently verified company/headline relevance check. Beware ambiguous ticker symbols.

Output is not committed to GitHub because the original dataset's redistribution permissions must be respected.

In [ ]:
MATCHED_NEWS = DATA / "matched_news.parquet"

if RUN_SOURCE_DOWNLOADS and NEWS_CSV.is_file() and PRICE_ZIP.is_file() and not MATCHED_NEWS.exists():
    import pyarrow as pa
    import pyarrow.parquet as pq

    with zipfile.ZipFile(PRICE_ZIP) as archive:
        symbols = {
            Path(path).stem.upper()
            for path in archive.namelist()
            if path.startswith("full_history/") and path.lower().endswith(".csv")
        }

    writer, saved = None, 0
    columns = ["Date", "Article_title", "Stock_symbol", "Url", "Publisher"]
    try:
        for chunk in pd.read_csv(
            NEWS_CSV, usecols=columns, dtype="string",
            chunksize=100_000, on_bad_lines="skip"
        ):
            chunk["Stock_symbol"] = chunk["Stock_symbol"].str.strip().str.upper()
            chunk = chunk.loc[chunk["Stock_symbol"].isin(symbols)].copy()
            chunk["Date"] = pd.to_datetime(chunk["Date"], errors="coerce", utc=True, format="mixed")
            chunk = chunk[
                chunk["Date"].notna()
                & chunk["Date"].dt.year.between(2007, 2020)
                & chunk["Article_title"].notna()
                & chunk["Article_title"].str.strip().ne("")
            ]
            if chunk.empty:
                continue
            table = pa.Table.from_pandas(chunk[columns], preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(str(MATCHED_NEWS), table.schema, compression="zstd")
            writer.write_table(table)
            saved += len(chunk)
    finally:
        if writer is not None:
            writer.close()
    print(f"Matched news rows: {saved:,}; saved to {MATCHED_NEWS}")
else:
    print("Matched-news export skipped (not enabled / inputs absent / output exists).")

## 5 · Optional project-level processing and regression checks

The original 46-cell history contained repeated `git pull`, import reloads and ad hoc smoke tests. Those are **not reproducible research steps**. We instead require an existing checkout, call the project code directly and make expensive operations explicit.

Historical selected ticker count was **144 requested**; this was an implementation-specific universe, not the same as the modern 135-ticker price cohort.

Do not run a new checkout's changed builder without recording its commit hash and inspecting protocol differences.

In [ ]:
if REPO.exists() and (REPO / ".git").is_dir():
    revision = subprocess.check_output(
        ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
    ).strip()
    print("Builder Git revision:", revision)
else:
    print("No project checkout. Clone the repository only when rebuilding.")

if RUN_TESTS:
    if not REPO.exists():
        raise FileNotFoundError("Clone MarketReactionLab into " + str(REPO))
    subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "tests/test_kaggle_pipeline.py"],
        cwd=REPO, check=True
    )

## 6 · Optional full historical dataset build

The builder receives the same **conservative date-only alignment protocol** and original 30-candle / 16-feature parameters. It requires extracted price CSVs and prepared news data. This is the expensive stage; it is **off** by default.

To avoid accidental data revision, we do not override existing datasets. A new output directory is used, and users must establish their own ticker universe provenance.

In [ ]:
if RUN_BUILD:
    if not REPO.exists():
        raise FileNotFoundError("Existing project checkout missing.")
    import sys
    sys.path.insert(0, str(REPO))
    from src.data.kaggle_dataset_builder import build_kaggle_research_dataset

    clean_news = DATA / "clean_news.parquet"
    prices = DATA / "raw_prices"
    assert clean_news.is_file() and prices.is_dir()
    assert "selected_tickers" in globals(), (
        "Provide the exact archived selected_tickers list explicitly before building; "
        "the original notebook generated 144 symbols using training-period headline coverage."
    )
    target = DATA / "processed_v2_conservative_rebuild"
    if target.exists():
        raise FileExistsError(f"Refusing to overwrite existing rebuild: {target}")

    manifest = build_kaggle_research_dataset(
        price_dir=prices, news_parquet_path=clean_news,
        output_dir=target, target_tickers=selected_tickers,
        candle_seq_len=30, label_threshold=0.005,
        train_end_date="2017-12-31", val_end_date="2018-12-31",
        test_end_date="2020-06-11", filter_retrospective=False,
        alignment_protocol="date_only_conservative_lag",
    )
    print(json.dumps(manifest, indent=2, default=str))
else:
    print("Full historical dataset rebuild disabled.")

## 7 · Audit already-saved V2 historical records

This is the practical **default execution path** when the historical Parquet files are mounted. We verify row and unique-session counts, label encoding, 16 feature columns, chronological ordering, unique sample IDs, and the `untrusted_date_only` flag.

**Do not interpret these structural checks as proof of real-world point-in-time headline availability.**

Do not hardcode a 282,203-row total: that number described a historical train/validation/test artifact and must not be confused with the **207,653 training+2018 news rows audited in V18**.

In [ ]:
paths = {
    "train_2009_2017": FULL_HISTORICAL_DIR / "train.parquet",
    "validation_2018": FULL_HISTORICAL_DIR / "validation.parquet",
}

audit_rows = []
for name, path in paths.items():
    if not path.is_file():
        print(name, "not mounted:", path)
        continue
    frame = pd.read_parquet(path)
    required = {"sample_id", "session_id", "reaction_label",
                "reference_date", "target_date", "timestamp_trust"}
    if not required.issubset(frame.columns):
        raise ValueError(f"{name}: missing {sorted(required - set(frame.columns))}")
    assert frame.sample_id.is_unique
    assert frame.reaction_label.isin([-1, 0, 1]).all()
    ref = pd.to_datetime(frame.reference_date).dt.date
    target = pd.to_datetime(frame.target_date).dt.date
    assert (ref < target).all()
    features = [c for c in frame.columns if c.startswith("feat_")]
    assert len(features) == 16, f"{name}: got {len(features)} features"
    audit_rows.append({
        "cohort": name,
        "article_rows": len(frame),
        "unique_sessions": int(frame.session_id.nunique()),
        "untrusted_date_only": int((frame.timestamp_trust == "untrusted_date_only").sum())
    })

if audit_rows:
    audit = pd.DataFrame(audit_rows)
    display(audit)
    print("Structural checks passed on the available files.")
else:
    print("No historical processed data attached; no historical audit executed.")

## 8 · Historical provenance and limitations

This notebook is a **cleaned documentation of the source pipeline**. It does not run FinBERT, XGBoost, Groq, V13 or V20 training, and it does not replace their archived scripts.

- Original prototype root README: **25 symbols / 12,871 samples / 19 features**.
- Subsequent V2 historical experiment: **165,983 training article rows**, **41,670 2018 validation article rows**; **84,584** and **18,178** unique sessions respectively; **16 indicators**.
- Later modern price pipeline: distinct 2020–2026 universe and feature construction; **not the same sample population**.
- Historical news publication timing: all **207,653** V18-inspected article records are labeled `untrusted_date_only`.
- After-close earnings and headlines can be incorrectly aligned by publication *date*, introducing look-ahead bias.
- Sample checks establish internal stored-data consistency only; they do **not** audit third-party historical price corrections, survivorship bias, or verified publication clocks.

See `notebooks/research/05_v20_frozen_models_audit.ipynb` for the frozen 2025–2026 price-model comparison. The two notebooks answer different questions.

**Academic disclosure:** Paper and notebook results are exploratory historical records, not peer-reviewed or verified trading results.